In [2]:
import pandas as pd

# Load dataset
df = pd.read_excel("/content/PPMI_Curated_Data_Cut_Public_20260511.xlsx")

# Genes to encode
genes = ["PRKN", "PINK1", "LRRK2", "GBA", "SNCA", "PARK7"]

# Add encoded gene columns
for gene in genes:
    df[gene] = df["subgroup"].str.contains(gene).astype(int)

# Check shape
print(df.shape)

# Display full dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

print(df)

Streaming output truncated to the last 5000 lines.
14450      1.0     0      0      1    0     0      0  
14451      1.0     0      0      1    0     0      0  
14452      1.0     0      0      1    0     0      0  
14453      NaN     0      0      0    0     0      0  
14454      0.0     0      0      0    0     0      0  
14455      0.0     0      0      0    0     0      0  
14456      0.0     0      0      0    0     0      0  
14457      NaN     0      0      0    0     0      0  
14458      NaN     0      0      0    0     0      0  
14459      NaN     0      0      0    0     0      0  
14460      NaN     0      0      0    0     0      0  
14461      2.0     1      0      0    0     0      0  
14462      2.0     1      0      0    0     0      0  
14463      2.0     1      0      0    0     0      0  
14464      2.0     1      0      0    0     0      0  
14465      2.0     1      0      0    0     0      0  
14466      NaN     0      0      0    0     0      0  
14467      NaN

In [3]:
print(df.shape)

(19450, 218)


In [7]:
from google.colab import files
files.download("PPMI_Curated_Data_with_Gene_Encoding.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import pandas as pd

# Load your current dataset
df = pd.read_excel("PPMI_Final_PD_ML_Dataset.xlsx")

# Remove RBD and Hyposmia columns
df = df.drop(columns=["RBD", "Hyposmia"])

# Save updated dataset
df.to_excel("PPMI_Gene_Only_Dataset.xlsx", index=False)

print("RBD and Hyposmia removed successfully!")
print(df.shape)

RBD and Hyposmia removed successfully!
(19450, 218)


In [9]:
gene_attributes = ['PRKN', 'PINK1', 'LRRK2', 'GBA', 'SNCA', 'PARK7']

# Calculate missing values for the specified gene attributes
missing_genes = df[gene_attributes].isnull().sum()
total_rows = len(df)
missing_percentage_genes = (missing_genes / total_rows) * 100

print("Missing data percentage for gene attributes:")
display(missing_percentage_genes.sort_values(ascending=False))

# Identify genes with more than 70% missing data
high_missing_genes = missing_percentage_genes[missing_percentage_genes > 70]

if not high_missing_genes.empty:
    print("\nGene attributes with more than 70% missing data:")
    display(high_missing_genes)
else:
    print("\nNo gene attributes have more than 70% missing data.")

Missing data percentage for gene attributes:


,0
PRKN,0.0
PINK1,0.0
LRRK2,0.0
GBA,0.0
SNCA,0.0
PARK7,0.0



No gene attributes have more than 70% missing data.


In [10]:
# Check for class imbalance in the 'Diagnosis' column
class_distribution = df['Diagnosis'].value_counts(normalize=True) * 100

print("Class distribution of 'Diagnosis' (%):")
display(class_distribution)


Class distribution of 'Diagnosis' (%):


,proportion
Diagnosis,
1,58.226221
0,41.773779


A significant difference in these percentages indicates class imbalance.

In [11]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from xgboost import XGBClassifier

# Define features (X) and target (y)
gene_attributes = ['PRKN', 'PINK1', 'LRRK2', 'GBA', 'SNCA', 'PARK7']
X = df[gene_attributes]
y = df['Diagnosis']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Original training set shape: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Original testing set shape: X_test {X_test.shape}, y_test {y_test.shape}")

Original training set shape: X_train (15560, 6), y_train (15560,)
Original testing set shape: X_test (3890, 6), y_test (3890,)


In [12]:
# Apply RandomOverSampler to the training data to handle class imbalance
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

print(f"Resampled training set shape: X_train_resampled {X_train_resampled.shape}, y_train_resampled {y_train_resampled.shape}")
print("Class distribution after oversampling:")
display(y_train_resampled.value_counts(normalize=True) * 100)

Resampled training set shape: X_train_resampled (18120, 6), y_train_resampled (18120,)
Class distribution after oversampling:


,proportion
Diagnosis,
0,50.0
1,50.0


In [13]:
# Initialize and train the XGBoost Classifier
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train_resampled, y_train_resampled)

print("XGBoost model trained successfully using the 6 gene attributes.")

XGBoost model trained successfully using the 6 gene attributes.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:15:20] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [14]:
from sklearn.metrics import classification_report, accuracy_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Model Evaluation on Test Set:")
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Evaluation on Test Set:
Accuracy Score: 0.7010282776349614

Classification Report:
               precision    recall  f1-score   support

           0       0.58      1.00      0.74      1625
           1       1.00      0.49      0.65      2265

    accuracy                           0.70      3890
   macro avg       0.79      0.74      0.70      3890
weighted avg       0.83      0.70      0.69      3890



## Results for Parkinson's Disease Prediction

Metric | Value
---|---
Cohort size | 19450 total (3890 in test set)
Train / test split | 80% / 20%, stratified
ROC-AUC (test) | Not calculated in this run (requires `predict_proba`)
PR-AUC (test) | Not calculated in this run (requires `predict_proba`)
Balanced accuracy (test) | 0.745
MCC (test) | Not calculated in this run (requires `matthews_corrcoef`)
Precision / Recall (class 1, Parkinson's) | 1.00 / 0.49
Precision / Recall (class 0, Healthy Control) | 0.58 / 1.00

In [15]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import balanced_accuracy_score

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Test-set confusion matrix (rows = actual, columns = predicted):")
print(f"Predicted: Healthy Control (0) | Predicted: Parkinson's (1)")
print(f"Actual: Healthy Control (0)        {tn}                         {fp}")
print(f"Actual: Parkinson's (1)           {fn}                         {tp}")

# Calculate balanced accuracy
balanced_acc = balanced_accuracy_score(y_test, y_pred)
print(f"\nBalanced Accuracy (test): {balanced_acc:.3f}")

Test-set confusion matrix (rows = actual, columns = predicted):
Predicted: Healthy Control (0) | Predicted: Parkinson's (1)
Actual: Healthy Control (0)        1625                         0
Actual: Parkinson's (1)           1163                         1102

Balanced Accuracy (test): 0.743


In [16]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred, output_dict=True)

precision_class_0 = report['0']['precision']
recall_class_0 = report['0']['recall']

precision_class_1 = report['1']['precision']
recall_class_1 = report['1']['recall']

print(f"Precision (Class 0, Healthy Control): {precision_class_0:.2f}")
print(f"Recall (Class 0, Healthy Control): {recall_class_0:.2f}")
print(f"Precision (Class 1, Parkinson's): {precision_class_1:.2f}")
print(f"Recall (Class 1, Parkinson's): {recall_class_1:.2f}")

Precision (Class 0, Healthy Control): 0.58
Recall (Class 0, Healthy Control): 1.00
Precision (Class 1, Parkinson's): 1.00
Recall (Class 1, Parkinson's): 0.49


Top features by importance: PRKN, PINK1, LRRK2, GBA, SNCA, PARK7 (as these were the only features used for training)